In [1]:
# enable cuda devices
import os
import warnings
import math

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

os.chdir('/app')
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
import time
import logging
import numpy as np
import torch
import json
import torch.nn as nn
import torchvision.transforms as transforms
from utils.common_utils import process_config
from datetime import datetime
from utils.strip_utils import compute_strip_importances, model_strip_group
from models import dataset
from utils.bit_config import bit_config_dict
from utils import misc

In [3]:

def train(train_loader, model, criterion, optimizer, epoch, configs):
    batch_time = AverageMeter('Time', ':6.3f')
    data_time = AverageMeter('Data', ':6.3f')
    losses = AverageMeter('Loss', ':.4e')
    top1 = AverageMeter('Acc@1', ':6.2f')
    top5 = AverageMeter('Acc@5', ':6.2f')
    progress = ProgressMeter(
        len(train_loader),
        [batch_time, data_time, losses, top1, top5],
        prefix="Epoch: [{}]".format(epoch))
        
    model.eval()
    end = time.time()
    for i, (images, target) in enumerate(train_loader):
        data_time.update(time.time() - end)

        images = images.cuda(0, non_blocking=True)
        target = target.cuda(0, non_blocking=True)

        # compute output
        output = model(images)
        loss = criterion(output, target)

        # measure accuracy and record loss
        acc1, acc5 = accuracy(output, target, topk=(1, 5))
        losses.update(loss.item(), images.size(0))
        top1.update(acc1[0], images.size(0))
        top5.update(acc5[0], images.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()

def validate(val_loader, model, criterion, configs):
    batch_time = AverageMeter('Time', ':6.3f')
    losses = AverageMeter('Loss', ':.4e')
    top1 = AverageMeter('Acc@1', ':6.2f')
    top5 = AverageMeter('Acc@5', ':6.2f')
    progress = ProgressMeter(
        len(val_loader),
        [batch_time, losses, top1, top5],
        prefix='Test: ')

    model.eval()

    with torch.no_grad():
        for i, (images, target) in enumerate(val_loader):
            images = images.cuda(0, non_blocking=True)
            target = target.cuda(0, non_blocking=True)

            # compute output
            output = model(images)
            loss = criterion(output, target)

            # measure accuracy and record loss
            acc1, acc5 = accuracy(output, target, topk=(1, 5))
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0], images.size(0))
            top5.update(acc5[0], images.size(0))
                    
    return top1.avg

def accuracy(output, target, topk=(1,)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self, name, fmt=':f'):
        self.name = name
        self.fmt = fmt
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)
    
class ProgressMeter(object):
    def __init__(self, num_batches, meters, prefix=""):
        self.batch_fmtstr = self._get_batch_fmtstr(num_batches)
        self.meters = meters
        self.prefix = prefix

    def display(self, batch):
        entries = [self.prefix + self.batch_fmtstr.format(batch)]
        entries += [str(meter) for meter in self.meters]
        print('\t'.join(entries))

    def _get_batch_fmtstr(self, num_batches):
        num_digits = len(str(num_batches // 1))
        fmt = '{:' + str(num_digits) + 'd}'
        return '[' + fmt + '/' + fmt.format(num_batches) + ']'

In [4]:
def compute_fisher_information(model, dataloader, criterion):
    model.eval()
    fisher_diagonal = {}
    
    for name, param in model.named_parameters():
        fisher_diagonal[name] = torch.zeros_like(param.data)
    
    for inputs, labels in dataloader:
        model.zero_grad()
        inputs, labels = inputs.cuda(), labels.cuda()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        for name, param in model.named_parameters():
            if param.grad is not None:
                fisher_diagonal[name] += param.grad.data ** 2
    
    # 取平均
    for name in fisher_diagonal:
        fisher_diagonal[name] /= len(dataloader)
    
    # 计算对角线元素之和
    total_fisher_information = 0.0
    for name in fisher_diagonal:
        total_fisher_information += fisher_diagonal[name].sum().item()
    
    return total_fisher_information

def compute_fisher_rao_distance(fisher1, fisher2, model1, model2, param_name_map):
    distance = 0.0
    for name1, param1 in model1.named_parameters():
        if name1 in param_name_map:
            name2 = param_name_map[name1]
            param2 = dict(model2.named_parameters())[name2]
            if name1 in fisher1 and name2 in fisher2:
                diff = (param1.data - param2.data).view(-1)
                fisher_avg = (fisher1[name1].view(-1) + fisher2[name2].view(-1)) / 2
                epsilon = 1e-8
                term = ((diff ** 2) * fisher_avg / (fisher_avg + epsilon)).sum().item()
                distance += term
    distance = distance ** 0.5
    return distance

In [5]:
def hessian_trace(model, val_loader, criterion, configs):
    importances, strip_importances_per_layer = compute_strip_importances(model, val_loader, criterion)
    strip_group = model_strip_group(model, importances, strip_importances_per_layer, configs.bits, configs.ratio, log=False)
    return strip_group

In [6]:
def compression_model(x, configs, original_model, train_loader, val_loader, criterion, inference_log_dir, net, num_classes):
    configs.ratio = x
    # Compute strip group based on Hessian trace (assuming this function adjusts the bitwidth configuration)
    strip_group = hessian_trace(original_model, val_loader, criterion, configs)

    # Build and load the compressed model
    if net == 'vgg11':
        from models.VGG import vgg11
        pre_trained_model = vgg11(num_classes)
        pre_trained_model.load_state_dict(torch.load(inference_log_dir))
        from models.Q_VGG import q_vgg11
        compressed_model = q_vgg11(pre_trained_model)
    elif net == 'vgg19':
        from models.VGG import vgg19
        pre_trained_model = vgg19(num_classes)
        pre_trained_model.load_state_dict(torch.load(inference_log_dir))
        from models.Q_VGG import q_vgg19
        compressed_model = q_vgg19(pre_trained_model)
    elif net == 'resnet18':
        from models.ResNet import resnet18
        pre_trained_model = resnet18(num_classes)
        pre_trained_model.load_state_dict(torch.load(inference_log_dir))
        from models.Q_ResNet import q_resnet18
        compressed_model = q_resnet18(pre_trained_model)
    elif net == 'resnet50':
        from models.ResNet import resnet50
        pre_trained_model = resnet50(num_classes)
        pre_trained_model.load_state_dict(torch.load(inference_log_dir))
        from models.Q_ResNet import q_resnet50
        compressed_model = q_resnet50(pre_trained_model)
    elif net == 'resnet20':
        from models.ResNet20 import resnet20
        pre_trained_model = resnet20(num_classes)
        pre_trained_model.load_state_dict(torch.load(inference_log_dir))
        from models.Q_ResNet20 import q_resnet20
        compressed_model = q_resnet20(pre_trained_model)
    else:
        raise ValueError("Unknown model type")

    # Apply bit configurations to the compressed model
    bit_config = strip_group
    name_counter = 0
    for name, m in compressed_model.named_modules():
        if name in bit_config.keys():
            name_counter += 1
            setattr(m, 'quant_mode', 'symmetric')
            setattr(m, 'bias_bit', configs.bias_bit)
            setattr(m, 'quantize_bias', (configs.bias_bit != 0))
            setattr(m, 'per_strip', True)
            setattr(m, 'name', f"Conv_{name_counter}")
            bitwidth = bit_config[name]
            setattr(m, 'weight_bit', bitwidth)

    torch.cuda.set_device(0)
    compressed_model = compressed_model.cuda(0)
    criterion = nn.CrossEntropyLoss().cuda(0)
    optimizer = torch.optim.SGD(compressed_model.parameters(), configs.lr,
                                momentum=configs.momentum,
                                weight_decay=configs.weight_decay)
    best_acc1 = 0

    # Train the compressed model
    print(configs.epochs)
    for epoch in range(configs.epochs):
        train(train_loader, compressed_model, criterion, optimizer, epoch, configs)
        acc1 = validate(val_loader, compressed_model, criterion, configs)
        is_best = acc1 > best_acc1
        best_acc1 = max(acc1, best_acc1)

    print(f"BEST ACC:{best_acc1}")
    return compressed_model


In [7]:
def compute_loss(x, configs, fisher_original, original_model, train_loader, val_loader, criterion, inference_log_dir, net, num_classes):
    print(f"ratio:{x}")
    compressed_model = compression_model(x, configs, original_model, train_loader, val_loader, criterion, inference_log_dir, net, num_classes)
  
    # Compute Fisher information for the compressed model
    fisher_compressed = compute_fisher_information(compressed_model, val_loader, criterion)

    # Compute loss as the squared difference between Fisher-Rao information
    loss = fisher_original - fisher_compressed
    return loss, fisher_compressed


In [8]:
configs = process_config('/app/configs/exp_for_cifar/iteration.json')
inference_log_dir = os.path.join(configs.logdir, configs.net, configs.dataset, 'best.pth')
net = configs.net

# Set up dataset
if configs.dataset == 'cifar10':
    train_loader, val_loader = dataset.get_cifar10(batch_size=configs.batch_size)
    num_classes = 10
elif configs.dataset == 'cifar100':
    train_loader, val_loader = dataset.get_cifar100(batch_size=configs.batch_size)
    num_classes = 100
else:
    raise ValueError("Unknown dataset type")

# Load the original model
if net == 'resnet18':
    from models.ResNet import resnet18
    original_model = resnet18(num_classes)
elif net == 'resnet50':
    from models.ResNet import resnet50
    original_model = resnet50(num_classes)
elif net == 'resnet20':
    from models.ResNet20 import resnet20
    original_model = resnet20(num_classes)
elif net == 'vgg11':
    from models.VGG import vgg11
    original_model = vgg11(num_classes)
elif net == 'vgg19':
    from models.VGG import vgg19
    original_model = vgg19(num_classes)
else:
    raise ValueError("Unknown model type")

original_model.load_state_dict(torch.load(inference_log_dir))
torch.cuda.set_device(0)
original_model = original_model.cuda(0)
criterion = nn.CrossEntropyLoss().cuda(0)

print(f"origin")
# normal_compressed_model = compression_model(0.5, configs, original_model, train_loader, val_loader, criterion, inference_log_dir, net, num_classes)
# Compute Fisher information for the original model
fisher_original = compute_fisher_information(original_model, val_loader, criterion)

# Gradient descent parameters
x = 0.9  # Starting compression ratio 
learning_rate = 0.0003
max_iters = 10
tol = 1e-3

print(f"Fisher-Rao before compression: {fisher_original}")
# for iter_num in range(max_iters):    
#     loss, fisher_compressed = compute_loss(x, configs, fisher_original, original_model, train_loader, val_loader, criterion, inference_log_dir, net, num_classes)
#     print(f"Fisher-Rao after compression:{fisher_compressed}")
#     if(loss > -100 and loss < 100):
#         loss = loss / 2 

#     x_new = x - learning_rate * loss
#     print(f"gradient:{learning_rate * loss}")
#     x_new = max(0.0, x_new)  # 确保 x_new 不小于 0
#     x_new = min(1.0, x_new) 
    

#     print(f"Iteration {iter_num+1}, Compression ratio: {x:.4f}, Loss: {loss:.4f}")

#     # if(loss > -15 and loss < 15):
#     #     break
    
#     x = x_new
#     print("-----------------------")


# print(f"Optimal compression ratio: {x}")
# print(f"Fisher-Rao before compression: {fisher_original}, after compression: {fisher_compressed}")

Files already downloaded and verified
Files already downloaded and verified
origin
Fisher-Rao before compression: 83.99298623739742


In [9]:
x = 0.74  # Starting compression ratio

normalompressed_model = compression_model(x, configs, original_model, train_loader, val_loader, criterion, inference_log_dir, net, num_classes)
fisher_compressed = compute_fisher_information(normalompressed_model, val_loader, criterion)

print(f"Optimal compression ratio: {x}")
print(f"Fisher-Rao before compression: {fisher_original}, after compression: {fisher_compressed}")

200
BEST ACC:81.44999694824219
Optimal compression ratio: 0.74
Fisher-Rao before compression: 83.99298623739742, after compression: 110.59561831783503
